<p style="text-align: left; font-size: 0.9em; margin-bottom: 0;">Bonifas Olivier <br>Filloux Louis <br> Gendronneau Maël</p>

<h1 style="text-align: center; margin-top: 0;">What Factors Determine Income? A Case Study on the Adult Dataset </h1>

---

## 1. Introduction

### 1.1 Project Objective

The primary objective of this project is to analyze the 1994 "Adult" Census dataset to identify key factors associated with income levels.

Based on this analysis, we will then build and evaluate a binary classification model to accurately predict whether an individual earns more or less than $50,000 per year. The success of this model will be measured not only by its accuracy but also by its ability to perform well on an imbalanced dataset.

### 1.2 The Dataset

This dataset, sourced from the UCI Machine Learning Repository, contains 15 features (a mix of demographic and employment-related variables) and one target variable, `income`.

### 1.3. Methodology and Plan

To achieve our objective, this analysis will follow a structured, step-by-step workflow:

1.  **Data Loading and Inspection:** We will load the dataset and perform an initial inspection to understand its structure, identify missing values, and check data types.
2.  **Exploratory Data Analysis (EDA):** This is the investigation phase. We will dive deep into the data to uncover trends, visualize distributions, and analyze the relationships between different features and the target variable (`income`).
3.  **Feature Engineering (FE):** Based *directly* on the insights from our EDA, we will clean, transform, and create new features to prepare a dataset that our models can understand and learn from effectively.
4.  **Modeling and Evaluation:** Finally, we will build several classification models (from a simple baseline to more complex ones), train them on our engineered data, and evaluate their performance using appropriate metrics to select the best one.

---

## 2. Data Loading and Inspection

In [ ]:
import warnings

import pandas as pd
import numpy as np
from optuna.exceptions import ExperimentalWarning

In [ ]:
df = pd.read_csv("adult.csv", sep=',')

In [ ]:
print("--- Info ---")
print(df.info())

The initial data inspection using `df.info()` reveals that the dataset consists of 48,842 entries, with 6 numerical columns and 9 categorical (object-type) columns.

Notably, pandas reports zero non-null values across all columns. This is highly unusual and suggests that missing values might be encoded with a special character ('?', 'N/A') rather than the standard `NaN` format. We will investigate this hypothesis next.

In [ ]:
df.value_counts("workclass")

The value_counts output for the workclass column confirms our earlier suspicion. It reveals a `?` category, which indicates that missing values are not encoded in a way that pandas can recognize automatically.
To rectify this, we will replace all occurrences of the `?` character with `np.nan` across the entire dataset. This will allow us to use standard pandas functions to analyze and handle these missing values.

In [ ]:
df = df.replace('?', np.nan)

In [ ]:
df.isnull().sum()

The count of these missing values reveals the following:

* `workclass`: 2799 missing values
* `occupation`: 2809 missing values
* `native-country`: 857 missing values

This is a significant amount of missing data (2799 out of 48 842 rows is about **5.7%** for `workclass`). Simply deleting these rows would cause a significant loss of information.

Our **Exploratory Data Analysis (EDA)** phase, which begins now, will need to determine the best strategy for handling these `NaN` values.

## Exploratory Data Analysis (EDA)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def analyze_univariate(feature_name, data=df):
    """
    Input : the name of the feature that we will analyze, and the dataframe which contains our feature

    Output : Different graph depending on the type of feature (Qualitative or Quantitative)
    
    The function works by verifying that the feature is in the dataset that was given, once this is done, it checks the feature's type.
    Then, depending on the type different graphs will be ploted :

    -If the data are numerical but not boolean (more than 2 types like age but not gender) a boxplot and an histogram will be ploted

    -If the data is a boolean, we simply plot the histogram of the feature

    -Lastly, if the data is neither boolean nor numerical, we plot a barplot to get the proportion of each class in the feature

    """
    
    if feature_name not in data.columns:
        raise ValueError("The feature name is not in the dataset")

    feature_type = data[feature_name].dtype
    fig = None
    
    is_numeric = pd.api.types.is_numeric_dtype(feature_type)
    is_bool = pd.api.types.is_bool_dtype(feature_type)

    if is_numeric and not is_bool:
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(f'{feature_name} Histogram', f'{feature_name} Boxplot')
        )

        fig.add_trace(
            go.Histogram(x=data[feature_name], name='Histogram'),
            row=1, col=1
        )

        fig.add_trace(
            go.Box(x=data[feature_name], name='Boxplot'),
            row=1, col=2
        )

        fig.update_layout(
            title_text=f'Univariate Analysis of: {feature_name}',
            showlegend=False
        )
    elif is_bool:
        fig = px.histogram(data, x=feature_name)
        fig.update_layout(yaxis_title='Count')
    else:
        data_counts = data[feature_name].value_counts().reset_index()
        data_counts.columns = [feature_name, 'count']
        total = data_counts['count'].sum()
        data_counts['percentage'] = (data_counts['count'] / total * 100)

        fig = px.bar(
            data_counts,
            x=feature_name,
            y='count',
            title=f'Distribution of: {feature_name}',
            text=data_counts['percentage'].apply(lambda x: f'{x:.1f}%')
        )

        fig.update_traces(textposition='outside')
        fig.update_layout(yaxis_title='Count')
        max_y_value = data_counts['count'].max()
        fig.update_layout(yaxis_range=[0, max_y_value * 1.15])

    if fig:
        fig.show()

In [ ]:
def analyze_bivariate(feature_name, target='income', data=df):
    """
    Input : the name of the feature that we will analyze, and the dataframe which contains our feature, 
    by default we analyze the income and the target together

    Output : Different graph depending on the type of feature (Qualitative or Quantitative)
    
    Once again, we verify that the feature is in the dataframe, then, if the feature is numerical (not boolean) we display a violon plot of the 
    feature and the target (we use income since this is our target but other feature can be used). If the feature is qualitative, we plot an 
    histogram to have an idea of the proportion of the target for each class in the feature.

    """

    if feature_name not in data.columns:
        raise ValueError(f"Feature '{feature_name}' is not in the dataset.")
    if target not in data.columns:
        raise ValueError(f"Target '{target}' is not in the dataset.")

    feature_type = data[feature_name].dtype
    fig = None

    is_numeric = pd.api.types.is_numeric_dtype(feature_type)
    is_bool = pd.api.types.is_bool_dtype(feature_type)

    if is_numeric and not is_bool:
        fig = px.violin(
            data,
            x=target,
            y=feature_name,
            color=target,
            box=True,
            title=f'Distribution of {feature_name} by {target}'
        )
    else:
        fig = px.histogram(
            data,
            x=feature_name,
            color=target,
            barmode='stack',
            barnorm='percent',
            title=f'Proportion of {target} by {feature_name}',
            text_auto='.1f'
        )
        fig.update_layout(yaxis_title='Count')

    if fig:
        fig.show()


### 3.1 Is our dataset imbalanced ?

In [ ]:
analyze_univariate('income')

**Observation:**
The visualization clearly shows that our target variable, `income`, is **significantly imbalanced**.

* The majority class, `<=50K`, accounts for approximately **76.1%** of the dataset.
* The minority class, `>50K`, makes up only **23.9%**.

**Implication:**
This imbalance is a critical finding and has two major consequences for our project:

1.  **Evaluation Metric:** Standard `accuracy` will be a misleading metric. A naive model that always predicts the majority class (`<=50K`) would achieve ~76% accuracy but would be completely useless. We must therefore use more robust metrics, such as the **F1-Score**, **AUC-ROC**, or **Precision/Recall**.

2.  **Modeling Strategy:** We may need to employ special techniques to help our model learn from the under-represented minority class. This could involve using class weights or applying a resampling technique like **SMOTE (Synthetic Minority Over-sampling Technique)** during the modeling phase.

### 3.2 How the missing values are distributed

In [ ]:
df_nan_workclass = df[df["workclass"].isnull()]

In [ ]:
analyze_univariate('income',data=df_nan_workclass)

We have identified that the `NaN` values in `workclass` are not random, they have a distinct income distribution (90.5% <=50K).

We must now decide on a handling strategy. A common method is to impute (replace) `NaN` values with the column's mode. We will test if this is a valid approach.

* **Hypothesis:** If the income distribution of the mode category is **similar** to our `NaN` group, imputation is a good strategy.
* **Alternative:** If the distribution is **different**, imputing with the mode would be incorrect and introduce bias. We would then need to consider another strategy, such as removing the rows or creating a new 'Unknown' category.

In [ ]:
df['workclass'].mode()

In [ ]:
df_private_workclass = df[df["workclass"] == 'Private']

In [ ]:
analyze_univariate('income',data=df_private_workclass)

Our analysis revealed a critical insight: the missing values are not random.

1.  **Finding 1:** The `NaN` group for `workclass` has a unique income distribution (**90.5% <=50K**), which is vastly different from the dataset's average (76.1% <=50K).
2.  **Finding 2:** Imputing with the mode ('Private', at 78.2% <=50K) would be incorrect. It would wrongly assign this distinct group to a category with a much higher earning rate, thereby damaging our model.

**Decision for Feature Engineering:**

Instead of deleting these rows (and losing ~5.7% of our data) or imputing incorrectly, our best strategy is to treat this "missingness" as an **informative feature**.

During the Feature Engineering phase, we will create a new, distinct category called **'Unknown'** (or similar) to replace these `NaN` values. This will allow our model to learn from this group, as their "unknown" status appears to be a strong predictor of lower income.

### 3.3 How the age affect the income ?

In [ ]:
analyze_univariate('age')

In [ ]:
analyze_bivariate('age')

In [ ]:
df_age = df[['age', 'income']].copy()
df_age['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)
df_age = df_age.groupby('age').mean()

In [ ]:
fig = px.scatter(
    x=df_age.index,
    y=df_age['income'],
    trendline='lowess',
    trendline_color_override='red'
)

fig.update_layout(
    title="Probability of winning >50K by age",
    xaxis_title="Ige",
    yaxis_title="Probability >50K"
)

fig.show()

#### Univariate Analysis
* **Observation:** The histogram and boxplot for `age` reveal a **right-skewed** distribution. This indicates that while most individuals are in the 25-45 age range, there is a long tail of older participants.
* **Initial Thought:** For a linear model, this skew might suggest a log transformation. However, we must analyze the relationship with the target first.

#### Bivariate Analysis
* **Observation (Violin Plot):** The violin plot clearly shows that `age` is a significant predictor of `income`. The distribution for the `>50K` group is shifted upwards, confirming that higher earners are, on average, older.
* **Observation (Logistic Plot):** The logistic probability plot (the scatter plot with the trendline) provides the most critical insight: the relationship between `age` and the probability of earning `>50K` is **distinctly non-linear**.
    1.  The probability is near-zero for young individuals (approx. <27).
    2.  It rises sharply through middle age, peaking around 50-60.
    3.  It then *decreases* for older individuals (approx. >70), likely due to retirement.

#### Conclusion & Feature Engineering Decision
A linear model would fail to capture this complex "rise-peak-fall" pattern. Giving the model the raw `age` (even scaled) or a simple log-transformed `age` would hide this information.

**Therefore, the most effective Feature Engineering strategy is binning (discretization).** By creating three distinct categories based on our findings ( **`<27`**, **`27-70`**, and **`>70`**), we will convert this non-linear relationship into a simple and powerful set of features that any model can easily understand.

## 3.4 How the workclass can affect the income ?

In [ ]:
analyze_univariate('workclass')

In [ ]:
analyze_bivariate('workclass')

#### Univariate Analysis
* **Observation:** The `workclass` feature is dominated by the **'Private'** category, which accounts for **73.6%** of all known job types.
* **Observation (Rarity):** The 'Without-pay' and 'Never-worked' categories are extremely rare, representing 0.0% of the data. This suggests they may be statistical noise or outliers.

#### Bivariate Analysis
* **Observation:** The relationship between `workclass` and `income` is highly significant.
    * **High Earners:** The **'Self-emp-inc'** category (self-employed, incorporated) is the strongest predictor of high income, with **55.3%** of individuals earning >50K. 'Federal-gov' also performs well (39.2% >50K).
    * **Low Earners:** 'Without-pay' (90.5% <=50K) and 'Never-worked' (100% <=50K) are, as expected, almost exclusively low-income.

#### Conclusion & Feature Engineering Decision
A critical insight emerges when comparing these findings to our separate analysis of the 2,799 `NaN` values (from section 3.2).

1.  **The (Misleading) Match:** The `NaN` group has an income distribution (90.5% <=50K) that is **identical** to the 'Without-pay' group.
2.  **The Logical Flaw:** It is tempting to impute all 2,799 `NaN` values as 'Without-pay'. However, this could be a mistake. It is highly unlikely that 5.6% of the dataset (`NaNs`) belongs to the rarest category ('Without-pay', at 0.0%). This statistical match is a coincidence, likely driven by a "floor effect" where both groups inherently have low income.
3.  **Decision:** Imputing with the mode ('Private') would also be incorrect, as their income distributions are very different (78.2% vs 90.5% <=50K).

The most robust and honest strategy is to treat the `NaN` group as its own distinct entity. Therefore, during Feature Engineering, we will create a new category, **'Unknown'**, to preserve this information.

### 3.5 What is the impact of fnlwgt on income ?

In [ ]:
analyze_univariate('fnlwgt')

In [ ]:
analyze_bivariate('fnlwgt')

**Univariate Analysis:**
* **Observation:** The `fnlwgt` (final weight) feature is heavily **right-skewed**, with a long tail of outliers, as seen in both the histogram and the boxplot.

**Bivariate Analysis:**
* **Observation:** The violin plot, which compares the distribution of `fnlwgt` for both `income` classes, is the most critical chart.
* **Finding:** The two violins (for `<=50K` and `>50K`) are **virtually identical**. They share the same median, interquartile range (IQR), and overall distribution shape.

**Conclusion & Feature Engineering Decision:**
The `fnlwgt` variable represents a statistical weight assigned by the census bureau (indicating how many real-world people this row represents) and is not an intrinsic feature of the individual.

Our bivariate analysis confirms this: the variable shows **zero predictive power** as its distribution is independent of the income class.

**Decision:** This feature is noise for our classification model. It will be **dropped** during the Feature Engineering phase to simplify the model and prevent it from learning from irrelevant data.

### 3.6 How the education affect the income ?

The dataset contains two education-related features: education (categorical) and education-num (numerical). We hypothesize that these two columns represent the same information, with education-num simply being an ordinal encoding of the education level.
To verify this, we will check if there is a consistent, one-to-one mapping between the values of these two columns. If they are found to be redundant, we will drop the categorical education column during the Feature Engineering phase to avoid data duplication and simplify our model.

In [ ]:
# Verify the mapping between 'education' and 'education-num'
education_mapping = df[['education', 'educational-num']].drop_duplicates().sort_values('educational-num')
print(education_mapping)

In [ ]:
analyze_univariate('educational-num')

In [ ]:
analyze_bivariate('educational-num')

**Redundancy Check (Conclusion from Screenshot):**
* **Finding:** The `education` (text) and `educational-num` (numeric) columns represent the same information. `educational-num` is a direct **ordinal** encoding of the diploma level (e.g., 9 = 'HS-grad', 13 = 'Bachelors').
* **Decision:** Keeping both is redundant. We will **drop the text `education` column** during Feature Engineering and use the pre-encoded `educational-num` column.

**Analysis of 'educational-num':**

**Univariate Analysis:**
* **Observation:** The histogram shows that `educational-num` is not a continuous variable, but an **ordinal** one. The distribution is multi-modal, with clear peaks at 9 ('HS-grad'), 10 ('Some-college'), and 13 ('Bachelors'), which correspond to the most common education levels.

**Bivariate Analysis:**
* **Observation (Violin Plot):** This is one of the strongest predictors in the dataset. There is a clear separation between the two income classes.
    * The `<=50K` group is centered around levels 9-10 (High School / Some College).
    * The `>50K` group is shifted significantly higher, centered around level 13 (Bachelors) and 14 (Masters).
* **Finding:** This confirms a strong, positive, monotonic relationship: **higher education level strongly correlates with a higher probability of earning >50K.**

**Conclusion & Feature Engineering Decision:**
While a tree-based model could use the `educational-num` (1-16) directly, the relationship for a linear model is not perfectly linear; it's based on "steps" or diploma tiers.

**Decision:** To capture this step-wise relationship, we will **bin** this ordinal feature into logical, real-world tiers during Feature Engineering. A potential grouping could be:
1.  **'Below-High-School'** (`<9`)
2.  **'High-School-Grad'** (`9-10`)
3.  **'College-Degree'** (`11-13`)
4.  **'Graduate-Degree'** (`>=14`)

This will create powerful, simple, and interpretable features for our models.

### 3.7 What is the impact of marital-status on income ?

In [ ]:
analyze_univariate('marital-status')

In [ ]:
analyze_bivariate('marital-status')

**Univariate Analysis:**
* **Observation:** The dataset is predominantly composed of two main groups: **'Married-civ-spouse' (45.8%)** and **'Never-married' (33.0%)**. The other categories are relatively small in comparison.

**Bivariate Analysis:**
* **Observation (The Key Insight):** This is perhaps the **strongest single predictor** in the dataset. The income distribution varies drastically across categories.
    * **'Married-civ-spouse'** is a powerful predictor of high income, with **44.5%** of individuals in this group earning `>50K`.
    * Conversely, **all other non-married statuses** (e.g., 'Never-married', 'Divorced', 'Widowed') are strong predictors of *low* income, with `>50K` earners making up less than 11% in each of those categories.

**Conclusion & Feature Engineering Decision:**
This feature is highly predictive and has a very low cardinality (only 7 categories).

**Decision:** No binning or regrouping is necessary or desirable, as this would destroy valuable information. This feature will be kept as-is and will be converted into new features using **One-Hot Encoding** during the Feature Engineering phase.

## 3.8 How the race affect the income ?

In [ ]:
analyze_univariate('race')

In [ ]:
analyze_bivariate('race')

**Univariate Analysis:**
* **Observation:** The dataset is predominantly **'White' (85.5%)**. 'Black' is the largest minority group (9.6%), while all other races make up less than 5% of the data.

**Bivariate Analysis:**
* **Observation (Key Insight):** The income proportion is not uniform across races. We can clearly identify two distinct clusters:
    1.  **High-Earning Cluster:** The 'White' (25.4% >50K) and 'Asian-Pac-Islander' (26.9% >50K) groups have very similar and relatively high-earning rates.
    2.  **Low-Earning Cluster:** The 'Black' (12.1% >50K), 'Amer-Indian-Eskimo' (11.7% >50K), and 'Other' (12.3% >50K) groups also share similar rates, but at a significantly lower level.

**Conclusion & Feature Engineering Decision:**
This feature is clearly predictive. While we could manually group the races based on these two clusters, it is not necessary.

**Decision:** The feature has a very **low cardinality** (only 5 categories). Grouping would result in a loss of information and is not required for model performance. We will therefore keep all 5 categories as-is and apply **One-Hot Encoding** during the Feature Engineering phase.

## 3.9 The impact of the native country

Before the analysis, let's look at the different values.

In [ ]:
print("Basic Information for Native-Country : ")
print(f"Total number of values: {len(df)}")
print(f"Number of unique country: {df['native-country'].nunique()}")
print(f"Missing Values (NaN): {df['native-country'].isnull().sum()}")
print("="*60)
print("Most represented country\n")
top_10_countries = df['native-country'].value_counts().head(10)
for country in top_10_countries.index :
    count = top_10_countries[country]
    pct = (count/len(df)) * 100
    print(f"{country}: {count} ({pct:.2f}%)")
print("="*60)
print("Least represented country\n")
top_10_countries = df['native-country'].value_counts().tail(5)
for country in top_10_countries.index :
    count = top_10_countries[country]
    pct = (count/len(df)) * 100
    print(f"{country}: {count} ({pct:.2f}%)")

We can notice that around 90% of the people in the dataset come from the United-States while the second most reprensented country is Mexico with only 1.95%. 

Since this feature is highly unbalanced, it will be more efficient to transform this feature in a boolean column named "usa-native-country" and we will consider that the nan values are non-united-state natives.

In [ ]:
bool_series = df['native-country'] == 'United-States'

df_seul = bool_series.to_frame(name='born_in_the_us')
df_seul['income'] = df['income']
print(df_seul.columns)
analyze_univariate('born_in_the_us', data=df_seul)

In [ ]:
analyze_bivariate('born_in_the_us',data=df_seul)

In [ ]:
native_income = pd.crosstab(df_seul['born_in_the_us'], df_seul['income'], normalize='index') * 100
#Here we use normalize index to have the percentage instead of the count

print(native_income.round(2))

## 3.10 The impact of Hours per week on income

In [ ]:
analyze_univariate('hours-per-week')

In [ ]:
analyze_bivariate('hours-per-week')

In [ ]:
df_temp = df[['hours-per-week', 'income']].copy()

df_temp['income_binary'] = (df_temp['income'] == '>50K').astype(int)

df_hours = df_temp.groupby('hours-per-week')['income_binary'].mean().to_frame(name='probability')

fig = px.scatter(
    x=df_hours.index,
    y=df_hours['probability'],  
    trendline='lowess',
    trendline_color_override='red'
)

fig.update_layout(
    title="Probability of Earning >50K by Hours Per Week",
    xaxis_title="Hours Per Week",
    yaxis_title="Probability >50K"
)

fig.show()

#### Univariate Analysis
* **Observation:** Those values makes sense as there is no legal limit of hours of work per weeks in the United-States (however 40 hours a week is known as a default value). It also explain why we have some values that are around 100 hours per weeks

#### Bivariate Analysis
* **Observation (Violin Plot):** The single most dominant feature of this plot is the massive, wide peak at 40 hours for both groups. This shows that the 40-hour work week is the standard for the vast majority of people, regardless of their income. For the >50K group (red violin), there is a very significant "bulge" or distribution of people working more than 40 hours. You can see thick, distinct peaks around 50, 60, and even 70 hours. For the <=50K group (blue violin), the plot is extremely thin above 40 hours. Very few people in this group work overtime.
* **Observation (Scatter Plot):** The logistic probability plot provides the most critical insight: the relationship between hours-per-week and the probability of earning >50K is distinctly non-linear. The probability is very low for part-time work (approx. < 40 hours) and rises slowly. It begins to rise sharply right around the 40-hour mark, corresponding to the "overtime" group we saw in the violin plot. The probability peaks around 50-60 hours and then appears to plateau or slightly decrease for those working extreme hours (approx. > 70 hours).

#### Conclusion & Feature Engineering Decision
Same as the age feature, a linear model would fail to capture this complex pattern. Giving the model the raw `hours per week` would hide this information.

**Therefore, the most effective Feature Engineering strategy is binning.** (<40 (Part-time), ==40 (Standard), and >40 (Overtime)), we convert this non-linear relationship into a simple and powerful set of features that any model can easily understand.

## 3.11 How the gender impact the income ?

In [ ]:
analyze_univariate('gender')

In [ ]:
analyze_bivariate('gender')

#### Univariate Analysis
* **Observation:** Here we can see that around a third of the people are women while the other two third are men, this is an important imbalance.

#### Bivariate Analysis
* **Observation (Histogram):** This chart clearly illustrates the difference in income distribution between genders, there is a stark disparity in the proportion of individuals earning >50K.
Males: 30.4% earn >50K.
Females: 10.9% earn >50K. 
In this dataset, males are nearly three times more likely to be in the high-income bracket than females, gender is a very strong predictor of income
(It makes even more sens knowing that those data come from a survey realised in 1991).

#### Conclusion & Feature Engineering Decision
Unloke other columns such as 'age' or 'hours per week', the gender variable is simple and its impact is direct. Therefore, using Label Encoding or One-Hot Encoding is the best option.

#### Univariate Analysis
* **Observation:** The histogram and boxplot clearly highlight the imbalance of the datas, around **90%** of the peoples are born in the usa (which makes sense since this is from a survey made in 1991 in the USA).
* **Initial Thought:** Even though this is an important imbalance, we want to keep this column as it will probably have a great impact.

#### Bivariate Analysis
* **Observation (Violin Plot):** Here again, the imbalance is really impactful, also, although the majority of individuals in both groups earn <=50K, the proportion of high earners (>50K) is higher among US-born individuals (24.40%) than among those not born in the US (19.82%).

#### Conclusion & Feature Engineering Decision
By converting the data in a boolean we will facilitate the model's complexity and avoid having useless or noisy features.

## 3.12 The impact of the relationship

In [ ]:
analyze_univariate('relationship')

In [ ]:
analyze_bivariate('relationship')

#### Univariate Analysis
* **Observation:** Here, we can see that husband is the most represented relationship, it makes sense because as we have seen previously, we have twice as much men than women. However, it is really interesting to note that there is only ~5% of wife (compared to 40% of husband). This raises questions about the data collection process and potential biases: are certain categories underrepresented, or does this reflect specific characteristics of the surveyed population ?

#### Bivariate Analysis
* **Observation (Histogram):** This plot reveals that relationship status is an extremely strong predictor of income. The 6 categories fall into two very distinct groups.
High-Income Group:
Wife: This group has the highest proportion of high-earners, with 46.9% in the >50K bracket.
Husband: This group is very similar, with 44.9% in the >50K bracket.

Low-Income Group:
All other categories show a very low probability of earning >50K.

#### Conclusion & Feature Engineering Decision
The data clearly indicates that being married is highly correlated with a high income, while all other relationship statuses are correlated with a low income. We have a categorical variable with 6 distinct levels. A linear model cannot use this data as-is, and simply one-hot encoding all 6 would create 6 separate features.
Our bivariate analysis shows that these 6 categories can be simplified into 2 groups that capture almost all the predictive power.
Therefore, the most effective Feature Engineering strategy is binning (or grouping).
By creating a new, single binary feature (e.g., is_married) that groups Husband and Wife into one category (e.g., 1) and all other statuses into a second category (e.g., 0).

## 3.13 How the occupation impact the income ?

In [ ]:
print("Basic Information for occupation : ")
print(f"Number of unique occupation: {df['occupation'].nunique()}")
print(f"Missing Values (NaN): {df['occupation'].isnull().sum()}")
print("="*60)
print("Most represented occupation\n")
print(df['occupation'].value_counts(normalize=True)*100)


In [ ]:
occupation_cleaned = df['occupation'].fillna('Unknown')

df_occupation_analysis = occupation_cleaned.to_frame(name='occupation')

df_occupation_analysis['income'] = df['income']

In [ ]:
analyze_univariate('occupation', data=df_occupation_analysis)

In [ ]:
analyze_bivariate('occupation', 'income', data=df_occupation_analysis)

#### Univariate Analysis
* **Observation:** 'occupation' is a high-cardinality feature, with 15 distinct categories. The top 5 categories are similarly sized, and the "Missing" group we created is significant at 5.8%.

#### Bivariate Analysis
* **Observation (Histogram):** The plot clearly shows that occupation is a very strong predictor of income. Unlike relationship which had two clear clusters, occupation shows a wide spectrum of probabilities for earning >50K. Exec-managerial (47.8%) and Prof-specialty (45.1%) have a very high proportion of high earners. A middle group, including Tech-support (29.0%), Sales (31.3%), and Craft-repair (22.6%), shows moderate success. A large group, including Other-service (4.1%), Handlers-cleaners (6.7%), and Priv-house-serv (1.2%), has a very low probability of earning >50K.

"Missing" Category: The "Missing" category we created has its own distinct income profile (9.4% earn >50K), separate from the lowest tiers.

#### Conclusion & Feature Engineering Decision
Based on this analysis, binning (grouping) is the wrong strategy for this feature. The most effective strategy is One-Hot Encoding (OHE). It would help us in :
Preserving Nuanced Information: occupation is not a simple linear feature. The 15 categories show a wide spectrum of income probabilities. Binning these into 3 or 4 groups would destroy this valuable, granular information.

Allowing for "Interaction Effects": This is the most important reason. occupation likely acts as a context for other features. For example, education might be the strongest predictor for a Prof-specialty.

hours-per-week might be the strongest predictor for a Craft-repair. If we bin these two jobs into a single "High-Income" group, the model loses the ability to learn these specific and powerful interaction effects.

Handling "Missing" Correctly: OHE will create a specific occupation_Missing feature. This allows the model to learn if the fact that an occupation is missing is, by itself, a predictive signal.

## 3.14 capital gain/loss

In [ ]:
fig_gain = px.histogram(df,
                        x='capital-gain',
                        title='Distribution of Capital Gain')

fig_gain.show()

fig_loss = px.histogram(df,
                        x='capital-loss',
                        title='Distribution of Capital Loss')

fig_loss.show()

This heavily skewed distribution is normal in this context and shows us that we need to change the data for our models.
We will treat capital gain and capital loss simultaneously in a new colum named capital change

In [ ]:
df_capital = df.copy()

df_capital['capital_net'] = df_capital['capital-gain'] - df_capital['capital-loss']

conditions = [
    (df_capital['capital_net'] > 0),
    (df_capital['capital_net'] < 0)
]

choices = ['Gain', 'Loss']

df_capital['capital_change'] = np.select(conditions, choices, default='NoChange')

df_capital = df_capital.drop(columns=['capital-gain', 'capital-loss', 'capital_net'])
df_capital['income'] = df['income']

In [ ]:
analyze_univariate('capital_change', data=df_capital)

In [ ]:
analyze_bivariate('capital_change', data=df_capital)

#### Univariate Analysis
* **Observation:** The feature is extremely imbalanced, the vast majority of individuals (87.1%) have no capital event ('NoChange'). A small minority experiences a 'Gain' (8.3%), and an even smaller group experiences a 'Loss' (4.7%). This confirms our feature engineering was successful. We have converted the sparse numerical data into a simple categorical feature.

#### Bivariate Analysis
* **Observation (Histogram):** This chart confirms that our strategy was a major success. The new capital_change feature is an extremely powerful predictor of income. The 'Gain' event is a massive indicator of high income, 61.7% of individuals who report a capital gain earn >50K. In a very interesting finding, the 'Loss' event is also a strong indicator of high income, 50.1% of this group earn >50K; the reason behind this might be that people with higher incomes are the ones who have disposable capital to invest in the first place. The majority group (87.1% of the data) acts as our baseline. Only 18.9% of this group earn >50K.

Conclusion: The analysis clearly shows that any capital event, whether it's a gain or a loss, is highly correlated with being in the high-income bracket, separating these individuals from the "NoChange" majority.

#### Conclusion & Feature Engineering Decision
We just need to preprocess on the dataframe and the feature will be complete. It will simply needs to be encoded (e.g., via One-Hot Encoding) to be used by the model. No further binning or transformation would be required.

## 3.15 Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=['int64', 'float64'])
corr_matrix = numeric_df.corr()

fig = px.imshow(corr_matrix,
                text_auto=True,
                aspect="auto",
                color_continuous_scale='Viridis',
                title='Heatmap de Corrélation des Features Numériques'
                )

fig.show()

**Observation:**
To conclude our Exploratory Data Analysis, we run a correlation heatmap on all numerical features to check for redundancy (multicollinearity).

The heatmap clearly shows that there are **no significant correlations** between any of the independent numerical features. All correlation coefficients are very close to zero (e.g., the highest being `educational-num` vs. `hours-per-week` at 0.14).

**Conclusion & Feature Engineering Decision:**
This is an excellent finding. It confirms that each numerical feature—such as `age`, `educational-num`, and `hours-per-week`—provides **unique, independent information** to the model.

**Decision:** We do not need to drop any of these features due to redundancy. Our analysis also confirms that `fnlwgt` is not correlated with any other feature, reinforcing our prior decision to drop it due to its lack of predictive power.

Our EDA is now complete. We have gathered all necessary insights to proceed with a well-justified Feature Engineering plan.

## 4. Feature Engineering (FE)

Based on our Exploratory Data Analysis, we will now construct two distinct feature engineering pipelines. This approach is necessary because linear models (e.g., Logistic Regression) and tree-based models (e.g., Random Forest) have different preprocessing requirements.

**1. Pipeline for Linear Models:**
This pipeline will involve a comprehensive set of transformations to meet the assumptions of linear models:
*   **NaN Imputation:** Replace missing values in `workclass` and `occupation` with a new 'Unknown' category, as decided in our EDA.
*   **Binning:** Discretize the `age` and `education-num` features into categorical bins to capture their non-linear relationships with income.
*   **Encoding:** Convert all categorical features (including the newly binned ones) into numerical format using One-Hot Encoding.
*   **Scaling:** Standardize all numerical features to have a mean of 0 and a standard deviation of 1. This is crucial to prevent features with larger scales from dominating the model.

**2. Pipeline for Tree-Based Models:**
This pipeline is simpler, as tree-based models are more robust:
*   **NaN Imputation:** Same as the linear pipeline (replace with 'Unknown').
*   **Encoding:** Convert all categorical features using One-Hot Encoding.
*   **No Binning/Scaling:** We will not apply binning or scaling. Tree models are not sensitive to the scale of features and are capable of capturing non-linear relationships on their own by finding optimal split points.

In [ ]:
X = df.drop('income', axis=1)
y = df['income']

In [ ]:
X

In [ ]:
X_linear = X.copy()
X_tree = X.copy()

### 4.1 Common Preprocessing Steps

First, we apply the transformations that are common to both the linear and tree-based model pipelines.

In [ ]:
cols_to_impute_common = ['workclass', 'occupation']

for col in cols_to_impute_common:
    X_linear[col] = X_linear[col].fillna('Unknown')
    X_tree[col] = X_tree[col].fillna('Unknown')

X_linear.drop(columns=['fnlwgt', 'education', ], inplace=True)
X_tree.drop(columns=['fnlwgt', 'education'], inplace=True)

### 4.2 Pipeline for Linear Models

Now, we apply the specific transformations for linear models: binning, grouping, encoding, and scaling.

In [ ]:
def create_capital_change(df):
    df['capital_change'] = 'NoChange'
    df.loc[df['capital-gain'] > 0, 'capital_change'] = 'Gain'
    df.loc[df['capital-loss'] > 0, 'capital_change'] = 'Loss'
    return df

X_linear = create_capital_change(X_linear)

age_bins = [0, 27, 71, 100]
age_labels = ['Young', 'Adult', 'Senior']
X_linear['age_binned'] = pd.cut(X_linear['age'], bins=age_bins, labels=age_labels, right=False)

edu_bins = [0, 9, 11, 14, 17]
edu_labels = ['Below-HS', 'HS-grad', 'College', 'Graduate']
X_linear['education_binned'] = pd.cut(X_linear['educational-num'], bins=edu_bins, labels=edu_labels, right=False)

hours_bins = [0, 40, 41, 100]
hours_labels = ['Part-time', 'Standard', 'Overtime']
X_linear['hours_binned'] = pd.cut(X_linear['hours-per-week'], bins=hours_bins, labels=hours_labels, right=False)

X_linear['is_married'] = X_linear['relationship'].isin(['Husband', 'Wife']).astype(int)

X_linear['is_from_usa'] = (X_linear['native-country'] == 'United-States').astype(int)

X_linear.drop(columns=['age', 'educational-num', 'hours-per-week', 'relationship', 'native-country', 'capital-gain', 'capital-loss'], inplace=True)

X_linear= pd.get_dummies(X_linear, drop_first=True)

### 4.3 Pipeline for Tree-Based Models

This pipeline is simpler: we only need to impute the final NaN column and then apply One-Hot Encoding.

In [ ]:
X_tree['native-country'] = X_tree['native-country'].fillna('Unknown')

X_tree = pd.get_dummies(X_tree, drop_first=True)


### 4.4 Target Variable Encoding

Finally, we encode the target variable `y` from categorical ('<=50K', '>50K') to numerical (0, 1), which is required by the models.

In [ ]:
y = y.apply(lambda x: 1 if x == '>50K' else 0)

## 5. Modeling and Evaluation

Now that our data is prepared, we can begin the modeling phase. We will start by creating a simple baseline model to establish a performance benchmark.

### 5.1 Baseline Model: Logistic Regression

A Logistic Regression is an excellent choice for a baseline because it is simple, fast, and interpretable. We will use the `X_linear` dataset, which has been specifically engineered for linear models.

The process is as follows:
1.  **Split Data**: We will split our data into training (80%) and testing (20%) sets. We use `stratify=y` to ensure that the proportion of income classes is the same in both sets, which is crucial for our imbalanced dataset.
2.  **Scale Data**: We will apply `StandardScaler` to the features. This is a critical step for linear models, as it ensures all features are on the same scale and prevents features with large values from dominating the model.
3.  **Train Model**: We will train the `LogisticRegression` model on the scaled training data.
4.  **Evaluate**: We will evaluate the model's performance on the unseen test data using a full suite of metrics suitable for imbalanced classification: the classification report (for Precision, Recall, F1-score), the Matthews Correlation Coefficient (MCC), and the ROC-AUC score.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_linear, X_test_linear, y_train, y_test = train_test_split(
    X_linear, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_linear, X_test_linear = scaler.fit_transform(X_train_linear), scaler.transform(X_test_linear)

X_train_tree, X_test_tree, _, _ = train_test_split(
    X_tree, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


These results provide an excellent and realistic baseline for our project. The scores confirm that our data leakage issue is resolved, and we can now analyze the model's true performance.


* **ROC-AUC: 0.8948**
    This is a very strong score. As seen in the graph, the ROC curve (blue line) is significantly "bowed" toward the top-left corner, far from the red random-chance line. This indicates that our model has **excellent discriminatory power** and can effectively distinguish between low-income and high-income individuals.

* **Matthews Correlation Coefficient (MCC): 0.5378**
    An MCC score well above 0 (and >0.5) is considered very solid. It confirms there is a strong positive correlation between our model's predictions and the true labels, and it's a reliable metric for an imbalanced dataset.

The classification report and confusion matrix clearly show the main weakness of this baseline model:

* **Low Recall for Class 1 (`>50K`): 0.57**
    This is the most critical metric. It means our model is only identifying **57%** of the individuals who *actually* earn over $50K.

* **Confirmation from Confusion Matrix:**
    This is confirmed by the high number of **False Negatives (1001)**. These are individuals who earn `>50K` ("True label" = 1) but were incorrectly predicted to earn `\<=50K` ("Predicted label" = 0). The model is "missing" almost half of the positive class.

* **Precision vs. Recall Trade-off:**
    The model has decent **Precision (0.71)** for Class 1, meaning when it *does* predict someone earns `>50K`, it's correct 71% of the time. However, it's too "cautious" and misses too many true positives to achieve this.


This Logistic Regression, serves as a strong **baseline**. We have proven that our features have predictive power (high AUC).

However, the model's linear nature is likely too simple to capture all the complex patterns that define the minority class, leading to a low **Recall**.

**Next Step:** The logical next step is to test a more complex, non-linear model (like a **RandomForest** or **XGBoost**). These models are designed to find complex interactions and should significantly improve our ability to find the 1001 high-income earners that this model missed.

a tester
XGboost
Naive Bayes
LightGBM
CatBoost
Tree
RandomForest
ExtraTrees
Log Reg
RidgeClassifier
SGDClassifier
AdaBoost
GradientBoosting
HistGradientBoosting
SVM
KNN


Voting
Bagging
Stacking



In [ ]:
from sklearn.model_selection import StratifiedKFold
from optuna.integration import OptunaSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import optuna
import warnings
from optuna.exceptions import ExperimentalWarning

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=ExperimentalWarning)

def optimize_and_fit_model(model, X_train, y_train, param_grid=None, scoring='f1_weighted', n_trials=30, timeout=300, cv=5, use_smote=False):
    """
    Input : The model that we will optimize and fit, the training set to fit to the model, the param grid for the OptunaSearch, 
    the type of scoring for the best parameters estimation, the number of trials for the search, the time limit in seconds for the 
    search of appropriate models, the number of folds in the cv splitter and finaly a boolean to check if we will use SMOTE before the 
    OptunaSearchCV

    Output : If a parameters grid wasn't provided, we simply return the model where the training data have been fitted, else, if the param grid
    is provided, we do an OptunaSearch using every parameter and we return the best parameters found and the model from those.

    

    """

    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    if use_smote:
        print("--- SMOTE Activated: Balancing classes inside CV folds ---")
        estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', model)
        ])
    else:
        estimator = model

    if param_grid and use_smote:
        new_grid = {}
        for key, dist in param_grid.items():
            new_key = f"clf__{key}" if not key.startswith("clf__") else key
            new_grid[new_key] = dist
        final_grid = new_grid
    else:
        final_grid = param_grid

    if final_grid:
        print(f"--- Starting Optuna Optimization ({n_trials} trials) ---")
        opt_search = OptunaSearchCV(
            estimator=estimator,
            param_distributions=final_grid,
            n_trials=n_trials,
            timeout=timeout,
            cv=skf,
            scoring=scoring,
            random_state=42,
            verbose=0
        )
        opt_search.fit(X_train, y_train)

        print(f"Best Params: {opt_search.best_params_}")
        print(f"Best Val Score ({scoring}): {opt_search.best_score_:.4f}")

        return opt_search.best_estimator_, opt_search

    else:
        print("--- No params provided. Fitting default model... ---")
        estimator.fit(X_train, y_train)
        return estimator, None

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, log_loss, classification_report

def compute_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)

    y_proba = None
    roc_auc = np.nan
    ll = np.nan

    try:
        if hasattr(model, "predict_proba"):
            y_proba_all = model.predict_proba(X_test)
            if y_proba_all.shape[1] == 2:
                y_proba = y_proba_all[:, 1]
                roc_auc = roc_auc_score(y_test, y_proba)
            else:
                y_proba = y_proba_all
                roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
            ll = log_loss(y_test, y_proba_all)
    except Exception as e:
        print(f"Warning: Could not calculate Probabilistic Metrics. Reason: {e}")

    global_metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_test, y_pred),
        "ROC AUC": roc_auc,
        "Log Loss": ll
    }
    global_df = pd.DataFrame([global_metrics])
    report_dict = classification_report(y_test, y_pred, output_dict=True)
    class_df = pd.DataFrame(report_dict).transpose()
    labels = [str(l) for l in sorted(list(set(y_test)))]
    class_df = class_df.loc[labels]

    return global_df, class_df, y_pred, y_proba

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, confusion_matrix

def visualize_performance(y_test, y_pred, y_proba=None, model_name="Model"):

    cm = confusion_matrix(y_test, y_pred)
    labels = sorted(list(set(y_test)))

    fig_cm = go.Figure(data=go.Heatmap(
        z=cm, x=labels, y=labels,
        hoverongaps=False, colorscale='Blues',
        text=cm, texttemplate="%{text}", textfont={"size": 16}
    ))
    fig_cm.update_layout(title=f"Confusion Matrix - {model_name}", xaxis_title="Predicted", yaxis_title="True")
    fig_cm.update_yaxes(autorange='reversed')
    fig_cm.update_xaxes(dtick=1)
    fig_cm.update_yaxes(dtick=1)
    fig_cm.show()

    if y_proba is not None and len(labels) == 2:
        fig = make_subplots(rows=1, cols=2, subplot_titles=("ROC Curve", "Precision-Recall Curve"))

        fpr, tpr, _ = roc_curve(y_test, y_proba)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='ROC', line=dict(color='blue')), row=1, col=1)
        fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random', line=dict(dash='dash', color='red')), row=1, col=1)

        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        fig.add_trace(go.Scatter(x=recall, y=precision, mode='lines', name='PR Curve', line=dict(color='blue')), row=1, col=2)

        fig.update_layout(title=f"Probabilistic Curves - {model_name}", height=500, showlegend=False)
        fig.show()


In [ ]:
def visualize_optimization(opt_search_object):
    if opt_search_object is None:
        return

    trials_df = opt_search_object.trials_dataframe()

    param_cols = [c for c in trials_df.columns if c.startswith('params_')]
    hover_data = {c: True for c in param_cols}

    fig = px.line(
        trials_df,
        x='number',
        y='value',
        markers=True,
        title='Optuna Optimization History',
        hover_data=hover_data
    )
    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, param_grid=None, scoring='f1', n_trials=30, timeout=180, use_smote=False):

    model_name = model.__class__.__name__
    if use_smote: model_name += " (with SMOTE)"

    trained_model, search_obj = optimize_and_fit_model(model, X_train, y_train, param_grid, scoring, n_trials, timeout=timeout, use_smote=use_smote)

    if search_obj:
        visualize_optimization(search_obj)

    global_df, class_df, y_pred, y_proba = compute_metrics(trained_model, X_test, y_test)

    print(f"\n=== Performance Report: {model_name} ===")

    print("\n--- Global Metrics ---")
    display(global_df.style.format("{:.4f}").background_gradient(cmap='Blues', axis=1))

    print("\n--- Per-Class Metrics ---")
    display(class_df.style.format("{:.4f}").background_gradient(cmap='Blues', axis=0))

    visualize_performance(y_test, y_pred, y_proba, model_name)


    return trained_model, global_df

In [ ]:
from optuna.distributions import IntDistribution, FloatDistribution, CategoricalDistribution

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(solver='liblinear', random_state=42)

param_grid_lr = {
    'C': FloatDistribution(1e-4, 100, log=True),
    'penalty': CategoricalDistribution(['l1', 'l2'])
}

best_lr, res_lr = evaluate_model(
    lr_model, X_train_linear, y_train, X_test_linear, y_test,
    param_grid=param_grid_lr, n_trials=15, timeout=60
)

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(probability=True, random_state=42)

param_grid_svm = {
    'C': FloatDistribution(0.1, 100, log=True),
    'gamma': CategoricalDistribution(['scale', 'auto'])
}

best_svm, res_svm = evaluate_model(
    svm_model, X_train_linear, y_train, X_test_linear, y_test,
    param_grid=param_grid_svm, n_trials=10, timeout=150
)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_jobs=-1)

param_grid_knn = {
    'n_neighbors': IntDistribution(3, 30),
    'weights': CategoricalDistribution(['uniform', 'distance']),
    'p': CategoricalDistribution([1, 2])
}

best_knn, res_knn = evaluate_model(
    knn_model, X_train_linear, y_train, X_test_linear, y_test,
    param_grid=param_grid_knn, n_trials=15, timeout=120
)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)

param_grid_dt = {
    'max_depth': IntDistribution(3, 20),
    'min_samples_split': IntDistribution(2, 20),
    'criterion': CategoricalDistribution(['gini', 'entropy'])
}

best_dt, res_dt = evaluate_model(
    dt_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_dt, n_trials=15, timeout=60
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid_rf = {
    'n_estimators': IntDistribution(50, 200),
    'max_depth': IntDistribution(5, 25),
    'min_samples_split': IntDistribution(2, 15)
}

best_rf, res_rf = evaluate_model(
    rf_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_rf, n_trials=15, timeout=180
)

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

et_model = ExtraTreesClassifier(random_state=42, n_jobs=-1)

param_grid_et = {
    'n_estimators': IntDistribution(50, 200),
    'max_depth': IntDistribution(5, 25),
    'min_samples_split': IntDistribution(2, 15),
    'criterion': CategoricalDistribution(['gini', 'entropy'])
}

best_et, res_et = evaluate_model(
    et_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_et, n_trials=15, timeout=180
)

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()

param_grid_nb = {
    'var_smoothing': FloatDistribution(1e-10, 1e-5, log=True)
}

best_nb, res_nb = evaluate_model(
    nb_model, X_train_linear, y_train, X_test_linear, y_test,
    param_grid=param_grid_nb, n_trials=15, timeout=60
)

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_model = SGDClassifier(loss='log_loss', random_state=42, n_jobs=-1)

param_grid_sgd = {
    'alpha': FloatDistribution(1e-5, 1e-1, log=True),
    'penalty': CategoricalDistribution(['l2', 'l1', 'elasticnet'])
}

best_sgd, res_sgd = evaluate_model(
    sgd_model, X_train_linear, y_train, X_test_linear, y_test,
    param_grid=param_grid_sgd, n_trials=15, timeout=60
)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(random_state=42)

param_grid_ada = {
    'n_estimators': IntDistribution(30, 150),
    'learning_rate': FloatDistribution(0.01, 1.0, log=True)
}

best_ada, res_ada = evaluate_model(
    ada_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_ada, n_trials=15, timeout=120
)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=42)

param_grid_gb = {
    'n_estimators': IntDistribution(50, 200),
    'learning_rate': FloatDistribution(0.01, 0.2, log=True),
    'max_depth': IntDistribution(3, 8)
}

best_gb, res_gb = evaluate_model(
    gb_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_gb, n_trials=10, timeout=180
)

In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(random_state=42, verbose=0)

param_grid_cat = {
    'iterations': IntDistribution(50, 300),
    'depth': IntDistribution(4, 10),
    'learning_rate': FloatDistribution(0.01, 0.3, log=True)
}

best_cat, res_cat = evaluate_model(
    cat_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_cat, n_trials=10, timeout=180
)

In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(random_state=42, verbosity=-1)

param_grid_lgbm = {
    'n_estimators': IntDistribution(50, 300),
    'num_leaves': IntDistribution(20, 100),
    'learning_rate': FloatDistribution(0.01, 0.3, log=True),
    'max_depth': IntDistribution(3, 12)
}

best_lgbm, res_lgbm = evaluate_model(
    lgbm_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_lgbm, n_trials=15, timeout=180
)

In [ ]:
from xgboost import XGBClassifier

xg_model = XGBClassifier(eval_metric='logloss', random_state=42)

param_grid_xgb = {
    'n_estimators': IntDistribution(50, 300),
    'max_depth': IntDistribution(3, 10),
    'learning_rate': FloatDistribution(0.01, 0.3, log=True),
    'subsample': FloatDistribution(0.6, 1.0)
}

best_xgb, res_xgb = evaluate_model(
    xg_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_xgb, n_trials=15, timeout=180
)

In [ ]:
from xgboost import XGBClassifier

xg_model = XGBClassifier(eval_metric='logloss', random_state=42)

param_grid_xgb = {
    'n_estimators': IntDistribution(50, 300),
    'max_depth': IntDistribution(3, 10),
    'learning_rate': FloatDistribution(0.01, 0.3, log=True),
    'subsample': FloatDistribution(0.6, 1.0)
}

best_xgb, res_xgb = evaluate_model(
    xg_model, X_train_tree, y_train, X_test_tree, y_test,
    param_grid=param_grid_xgb, n_trials=15, timeout=180, use_smote=True
)